In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
DATA_DIR = '/kaggle/input/competitions/store-sales-time-series-forecasting'

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')
oil = pd.read_csv(f'{DATA_DIR}/oil.csv')
holidays = pd.read_csv(f'{DATA_DIR}/holidays_events.csv')

In [ ]:
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

In [ ]:
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

In [ ]:
full_dates = pd.DataFrame({'date': pd.date_range('2013-01-01', '2017-08-31')})


oil_full = pd.merge(full_dates, oil, on='date', how='left')
oil_full['dcoilwtico'] = oil_full['dcoilwtico'].interpolate(method='linear').bfill().ffill()


train = train.merge(oil_full, on='date', how='left')
test = test.merge(oil_full, on='date', how='left')

In [ ]:
real_holidays = holidays[~holidays['transferred']].copy()

real_holidays = real_holidays.drop_duplicates(subset=['date'])[['date', 'type']]
real_holidays = real_holidays.rename(columns={'type': 'holiday_type'})

In [ ]:
train = train.merge(real_holidays, on='date', how='left')
test = test.merge(real_holidays, on='date', how='left')

In [ ]:
train['holiday_type'] = train['holiday_type'].fillna('WorkDay')
test['holiday_type'] = test['holiday_type'].fillna('WorkDay')

In [ ]:
for df in [train, test]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek                  # 0: 월요일 ~ 6: 일요일
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)         # 토/일은 1, 평일은 0
    df['is_payday'] = df['day'].isin([15, 30, 31]).astype(int)

In [ ]:
cat_features = ['family', 'city', 'state', 'type', 'holiday_type']

In [ ]:
for col in cat_features:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [ ]:
import numpy as np

In [ ]:
train_df = train[train['date'] >= '2016-01-01'].copy()

In [ ]:
feature_cols = [
    'store_nbr', 'family', 'onpromotion', 
    'city', 'state', 'type', 'cluster', 
    'dcoilwtico', 'holiday_type',
    'year', 'month', 'day', 'dayofweek', 'is_weekend', 'is_payday'
]

In [ ]:
val_cutoff = train_df['date'].max() - pd.Timedelta(days=15)

train_split = train_df[train_df['date'] < val_cutoff]
val_split = train_df[train_df['date'] >= val_cutoff]

In [ ]:
X_train = train_split[feature_cols]
y_train = np.log1p(train_split['sales'])

X_val = val_split[feature_cols]
y_val = np.log1p(val_split['sales'])

X_test = test[feature_cols]

In [ ]:
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

val_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

val_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
)

val_preds_log = val_model.predict(X_val)
val_preds = np.clip(np.expm1(val_preds_log), 0, None)
val_rmsle = root_mean_squared_error(y_val, np.log1p(val_preds))

print(f"\n[8단계 완료] 최적 트리 수: {val_model.best_iteration_}")
print(f"[8단계 완료] Validation RMSLE: {val_rmsle:.4f}")

In [ ]:
# 1. 2016년 전체 학습 데이터 정의
X_full = train_df[feature_cols]
y_full = np.log1p(train_df['sales'])

# 2. 검증에서 찾은 최적 트리 수 가져오기 (없을 경우 기본값 500)
best_trees = val_model.best_iteration_ if val_model.best_iteration_ is not None else 500
print(f"최종 모델에 적용할 트리 개수: {best_trees}")

# 3. 전체 데이터로 최종 모델 학습
final_model = lgb.LGBMRegressor(
    n_estimators=best_trees,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full, y_full)

# 4. Test 데이터 예측 및 원래 단위로 복원 (expm1)
test_preds_log = final_model.predict(X_test)
test_preds = np.clip(np.expm1(test_preds_log), 0, None)

# 5. submission.csv 생성
submission = pd.DataFrame({
    'id': test['id'],
    'sales': test_preds
})

submission.to_csv('submission.csv', index=False)

print("\n[9단계 완료] submission.csv 생성 완료!")
print(f"결측치 수: {submission.isnull().sum().sum()}")
print(f"행/열 크기: {submission.shape}")
print(submission.head())